### Load MNIST

The MNIST (numbers) dataset is built into PyTorch and can be loaded from using the datasets class.

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
test_set = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

### Prepare the Data

In [ ]:
train_x = train_set.data.reshape((60000, 28*28))
train_x = train_x.float() / 255
test_x = test_set.data.reshape((10000, 28 * 28))
test_x = test_x.float() / 255

In [ ]:
num_classes = train_set.targets.max() + 1
train_y = F.one_hot(train_set.targets, num_classes=num_classes).float()
test_y = F.one_hot(test_set.targets, num_classes=num_classes).float()

### Build Network

In [ ]:
# Define the neural network model as a subclass of nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self, no_layers):
        super(NeuralNetwork, self).__init__()
        self.hidden_layers = nn.ModuleList([nn.Linear(28*28, 200)] + [nn.Linear(200, 200) for _ in range(no_layers)])
        self.output_layer = nn.Linear(200, 10)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = torch.relu(layer(x))
        x = self.output_layer(x)
        return x

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_x = train_x.to(device)
train_y = train_y.to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

test_x = test_x.to(device)
test_y = test_y.to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

### Train

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
layers = [1, 5, 9]
no_epochs = 100
history_dict = dict()

In [ ]:
for ilr in layers:
    model = NeuralNetwork(ilr).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    key= str(ilr+1) + ' layers'
    history_dict[key] = train_model(model, train_loader, test_loader, loss_fn, optimizer, no_epochs)

### Plot Training and Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))

marker_list = ['^', '.', 'x', '+', 'v',  '*', 'o']

for i, ilr in enumerate(history_dict):
    acc_hist = history_dict[ilr]
    acc_values = acc_hist['train_acc']
    val_acc_values = acc_hist['test_acc']
    marker = marker_list[i]
    ax.plot(epochs, acc_values, color='k', marker=marker, linestyle='-', label = 'Training ' + ilr)
    ax.plot(epochs, val_acc_values, color='k', marker=marker, linestyle=':', label = 'Validation ' + ilr)

ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy')
ax.legend()
ax.set_ylim(92,)
fig.savefig('TestMNISTOverUnder.png', dpi=300, bbox_inches='tight')